# 📈 Correlación y regresión lineal simple · No resuelto

**Métodos Cuantitativos II · Antropología UAH · 11 de diciembre de 2026**

## 🌍 Antes de abrir la base: ¿qué queremos investigar?

Hacia 1888, Suiza atravesaba una **transición demográfica**: la fecundidad comenzaba a descender, pero los cambios no ocurrían del mismo modo en todos los territorios. Esta base reúne datos de **47 distritos francófonos** y permite preguntar: **¿cómo se relacionaban las diferencias de fecundidad entre distritos con sus condiciones sociales y económicas?** [Fuente y descripción de la base](https://stat.ethz.ch/R-manual/R-devel/library/datasets/html/swiss.html).

La mirada sociológica consiste en poner las diferencias demográficas en contexto. Por ejemplo, los distritos podían diferir en el acceso a la educación y en sus formas de trabajo. Si vemos que esas características van de la mano con la fecundidad, tenemos una pista para investigar cómo se organizaban la vida familiar y el trabajo en esos lugares. El gráfico por sí solo no explica por qué aparece el patrón.

| Variables que cruzaremos | ¿Para qué sirve compararlas? |
|:---|:---|
| **Escolaridad y fecundidad** | Para ver si los distritos con mayor proporción de reclutas que estudiaron más allá de la primaria tienden a presentar mayor o menor fecundidad. La escolaridad medida es la de los reclutas, no la de toda la población. |
| **Trabajo agrícola y fecundidad** | Para ver si el patrón de fecundidad cambia entre distritos donde una proporción mayor o menor de los hombres se dedica a la agricultura. Podemos pensar si la organización del trabajo y de los hogares ayuda a entender ese patrón. |
| **Mortalidad infantil y fecundidad** | Para explorar si los distritos donde mueren más niños antes de cumplir un año también muestran diferencias de fecundidad. Podemos preguntarnos si condiciones compartidas, como el saneamiento o el acceso a cuidados, se relacionan con ambos indicadores. |

Aquí **cruzar** significa mirar **dos variables juntas para los mismos distritos**. Primero ubicamos cada distrito como un punto, luego resumimos la asociación y, por último, estimamos una recta para expresar cuánto cambia la fecundidad ajustada al comparar distritos con distintos valores del predictor.

📌 **Qué podemos concluir:** hablamos de diferencias **entre distritos**, no de lo que hizo cada persona o familia. Tampoco podemos afirmar que la escolaridad, la agricultura o la mortalidad infantil causaron un cambio en la fecundidad. Para investigar explicaciones harían falta otras fuentes, como censos, registros históricos o estudios sobre la vida familiar de la época.

💬 **Antes de programar:** elige uno de los cruces. ¿Qué patrón esperarías ver? ¿Qué otra característica del distrito buscarías para interpretar ese resultado?

Al terminar podrás describir una asociación con un gráfico, comparar tres coeficientes de correlación, interpretar una regresión lineal y distinguir una predicción dentro del rango de los datos de una extrapolación.

🧭 **Cómo trabajar:** ejecuta los bloques en orden. Antes de cada bloque de análisis, escribe qué esperas observar; después, explica el resultado con palabras y unidades. En Colab, usa un entorno de ejecución **R**.

## 📂 0. Subir el archivo de datos

1. Ten a mano el archivo **`swiss.csv`** entregado para esta clase.
2. En Colab, pulsa **Conectar** y abre **Archivos** (ícono de carpeta a la izquierda).
3. Pulsa **Subir al almacenamiento de la sesión** y selecciona `swiss.csv`. Comprueba que aparezca en el panel con ese nombre.
4. Ejecuta la primera celda de R. Deberías ver seis filas de la base. Si inicias una nueva sesión y el archivo ya no aparece, vuelve a subirlo.

## 🔎 1. Conocer los datos

Cada fila representa uno de **47 distritos francófonos de Suiza** hacia **1888**. `Canton` conserva el nombre de la columna del CSV original; aquí identifica al distrito. Los datos son agregados: un patrón entre distritos no describe necesariamente lo que ocurre entre personas.

| Variable | Qué mide |
|:---|:---|
| `Fertility` | Índice estandarizado de fecundidad; no es un número de hijos ni un porcentaje. |
| `Education` | Porcentaje de reclutas con educación posterior a la primaria. |
| `Agriculture` | Porcentaje de hombres ocupados en agricultura. |
| `Catholic` | Porcentaje de población católica. |
| `Examination` | Porcentaje de reclutas con la mejor calificación en el examen militar. |
| `InfantMortality` | Porcentaje de nacidos vivos que fallecieron antes de cumplir un año. |

In [ ]:
# Importamos el CSV que subiste al almacenamiento de la sesión.
archivo <- if (dir.exists("/content")) "/content/swiss.csv" else "swiss.csv"
if (!file.exists(archivo)) {
  stop("No encuentro swiss.csv. Súbelo desde Archivos en Colab y vuelve a ejecutar esta celda.", call. = FALSE)
}
suiza <- read.csv(archivo, stringsAsFactors = FALSE)
columnas <- c("Canton", "Fertility", "Agriculture", "Examination", "Education", "Catholic", "InfantMortality")
if (!all(columnas %in% names(suiza))) {
  stop("El archivo no tiene las columnas esperadas. Revisa que sea el swiss.csv de la clase.", call. = FALSE)
}

# Si ggplot2 ya está disponible, este bloque no instala nada.
if (!requireNamespace("ggplot2", quietly = TRUE)) {
  install.packages("ggplot2", repos = "https://cloud.r-project.org")
}
library(ggplot2)
theme_set(theme_minimal(base_size = 13))

head(suiza)

In [ ]:
# dim() cuenta filas y columnas; summary() resume los valores observados.
# Min. y Max. muestran el valor más bajo y más alto de cada variable.

dim(suiza)
summary(suiza[c("Education", "Fertility", "Agriculture")])
colSums(is.na(suiza))
# Rango de escolaridad: el primer número es el mínimo; el segundo, el máximo.
range(suiza$Education)

🔎 **¿Qué nos dice el rango?** `Education` es el porcentaje de reclutas con educación posterior a primaria en cada distrito. El mínimo y el máximo son los porcentajes más bajo y más alto que sí observamos. Sirven para describir los datos y reconocer si una comparación queda fuera de lo observado. No hace falta memorizar esos números.

## 👀 2. Mirar antes de calcular

Cada punto es un distrito. Ponemos la escolaridad en el eje horizontal y la fecundidad en el vertical. Lee la nube de izquierda a derecha: ¿sube o baja? ¿Se parece a una recta? ¿Hay puntos apartados?

In [ ]:
# Cada punto es un distrito. X = escolaridad; Y = fecundidad.
# Lee primero la dirección de la nube: ¿tiende a subir o a bajar?

nube <- ggplot(suiza, aes(x = Education, y = Fertility)) +
  labs(x = "Reclutas con educación posterior a primaria (%)",
       y = "Índice estandarizado de fecundidad")
nube + geom_point(color = "#176B73", size = 2.5) +
  labs(title = "Escolaridad y fecundidad por distrito")

In [ ]:
# Catholic cambia el tamaño de los puntos; seguimos usando los mismos ejes.
# Los nombres se muestran solo para algunos distritos para facilitar la lectura.

destacados <- subset(suiza, Education >= 30)
nube + geom_point(aes(size = Catholic), color = "#176B73", alpha = 0.7) +
  geom_text(data = destacados, aes(label = Canton),
            nudge_y = 3, check_overlap = TRUE, size = 3) +
  scale_size_area(max_size = 7) +
  labs(title = "Una tercera variable: composición religiosa",
       size = "Población católica (%)")

Cambiar el tamaño de los puntos ayuda a formular preguntas sobre una tercera variable. No equivale a controlarla estadísticamente. ¿Qué procesos históricos podrían relacionar educación y fecundidad? ¿Qué datos necesitarías para evaluarlos?

## 📊 3. Medir la asociación

- **Pearson (`r`)** resume qué tan alineados están los puntos con una recta. Puede ser sensible a observaciones extremas.
- **Spearman (`rho`)** aplica la correlación a los rangos. Puede detectar una tendencia que siempre sube o baja aunque no sea recta.
- **Kendall (`tau`)** compara pares: si dos distritos se ordenan en el mismo sentido en ambas variables, el par es concordante; si se ordenan al revés, es discordante.

Los tres van de −1 a 1. El signo indica la dirección. Un valor cercano a cero indica poca asociación del tipo que mide ese coeficiente. Sus magnitudes no son directamente intercambiables.

In [ ]:
# cor() devuelve el coeficiente, sin prueba ni intervalo.
# Compara el signo; las magnitudes de los tres métodos no son equivalentes.

metodos <- c("pearson", "spearman", "kendall")
coeficientes <- sapply(metodos, function(metodo) {
  cor(suiza$Education, suiza$Fertility, method = metodo)
})
round(coeficientes, 3)

### 🧪 Pruebas de correlación

Completa las tres pruebas. En cada salida identifica `sample estimates`, `p-value` y la hipótesis alternativa. Busca el intervalo del 95 % de Pearson: ¿incluye cero? ¿El intervalo se refiere a la correlación o a la pendiente?

**Pista:** `t`, `S` y `z` son estadísticos de prueba, no coeficientes. La prueba es bilateral: la alternativa es que el coeficiente sea distinto de cero.

In [ ]:
# cor.test() muestra el coeficiente y el valor p de la prueba bilateral.
# En Pearson, el intervalo que aparece es el de la correlación, no el de la pendiente.
# exact = FALSE pide un valor p aproximado para las pruebas por rangos con empates.

# Completa el método en cada línea y quita el primer # para ejecutar.
# cor.test(~ Education + Fertility, data = suiza, method = "___")
# cor.test(~ Education + Fertility, data = suiza, method = "___", exact = FALSE)
# cor.test(~ Education + Fertility, data = suiza, method = "___", exact = FALSE)
# Mis resultados: signo, valor p e intervalo cuando esté disponible.

🗣️ **Discute:** un valor p pequeño no mide la fuerza de la relación ni identifica una causa. La interpretación inferencial requiere supuestos como independencia entre observaciones; distritos vecinos podrían estar relacionados. Estos resultados son un ejercicio con datos históricos, no una generalización a cualquier lugar o época.

## 📐 4. Ajustar una regresión lineal simple

La correlación es simétrica. Para predecir con una regresión, en cambio, elegimos un resultado ($Y$) y un predictor ($X$). En `lm(Y ~ X)` la variable a la izquierda es la que queremos explicar o predecir. Mínimos cuadrados elige la recta que minimiza la suma de los residuos al cuadrado. Cada residuo es el valor observado menos el ajustado.

### 📍 Antes de ajustar: ¿qué es un residuo?

Imagina que un distrito registra **60 puntos** de fecundidad y la recta estima **70**. Comparamos dos valores para el **mismo distrito** y la **misma escolaridad**: el punto observado y el punto de la recta. La diferencia vertical entre ellos, con signo, es el **residuo**.

| Residuo = observado − ajustado | ¿Dónde queda el punto observado? | ¿Qué hizo la recta? |
|:---|:---|:---|
| **Positivo** | Por encima de la recta | Estimó de menos. |
| **Negativo** | Por debajo de la recta | Estimó de más. |
| **Cero** | Sobre la recta | Acertó el valor observado. |

El signo indica **en qué dirección** se apartó la estimación; la distancia a cero indica **por cuántos puntos**. Ahora probemos con los números del ejemplo:

In [ ]:
# Ejemplo inventado: el dato observado fue 60, pero la recta estimó 70.
observado_ejemplo <- 60
ajustado_ejemplo <- 70
observado_ejemplo - ajustado_ejemplo
# Da -10: la recta estimó 10 puntos de más.
# Antes de seguir: si intercambiamos 60 y 70, ¿qué signo tendría el residuo?

In [ ]:
# En lm(Y ~ X), Y va a la izquierda y X a la derecha.
# summary(): lee la pendiente en la fila Education, columna Estimate.
# confint(): busca los límites del 95 % en la misma fila. ¿Queda el cero entre ellos?

# modelo_educacion <- lm(___ ~ ___, data = suiza)
# print(summary(modelo_educacion))
# print(confint(modelo_educacion))

### 🎯 Interpretar el intervalo de la pendiente

Lee la fila `Education` de `confint()`. Anota el límite inferior y el superior. **¿Incluye cero?** Explica qué significa el resultado para un aumento de un punto porcentual de escolaridad. Distingue este intervalo del que viste en la prueba de Pearson.

In [ ]:
# La línea es el valor ajustado por el modelo para cada nivel de escolaridad.
# La banda corresponde a la media estimada, no a un distrito individual.

nube + geom_point(color = "#176B73", size = 2.5) +
  geom_smooth(method = "lm", formula = y ~ x, color = "#A14225",
              fill = "#F1C4AE", se = TRUE) +
  labs(title = "Recta estimada con escolaridad",
       subtitle = "La banda estima la incertidumbre de la media ajustada")

### 📍 Calcula un residuo

Busca el primer distrito. Compara la fecundidad observada con la estimada por la recta. **Residuo = observado − ajustado.** ¿Qué signo tiene? ¿La recta subestima o sobreestima ese distrito?

In [ ]:
# suiza[1, c("Canton", "Fertility")]
# fitted(modelo_educacion)[1]
# mi_residuo <- ___ - ___
# mi_residuo

In [ ]:
# Eje X: fecundidad ajustada. Eje Y: observado menos ajustado.
# Arriba de cero: la recta quedó corta. Abajo: la recta estimó de más.

# Ejecuta este gráfico después de completar modelo_educacion.
# plot(modelo_educacion, which = 1)

En el gráfico de residuos, mira si los puntos están dispersos alrededor del cero o si forman una curva o un abanico. ¿Qué te dice eso sobre la recta? Contrasta lo que ves con el bloque `Residuals` de `summary()`.

### 🌾 Cambiar el predictor

Ajusta un modelo de fecundidad con agricultura y compara su pendiente y R² con el modelo de escolaridad.

In [ ]:
# Cambiamos el predictor a Agriculture, manteniendo Fertility como resultado.
# Lee Residuals para ver cuánto se apartan los distritos de esta recta.
# La pendiente y su intervalo están en la fila Agriculture, no en (Intercept).

# modelo_agricultura <- lm(___ ~ ___, data = suiza)
# summary(modelo_agricultura)
# confint(modelo_agricultura)

### 🧾 Lee los residuos del modelo de agricultura

1. En **Residuals**, busca el mínimo y el máximo. ¿En cuál de esos dos casos la recta estimó de más? ¿En cuál estimó de menos?
2. **1Q** y **3Q** delimitan la mitad central de los residuos. ¿Una mediana cercana a cero significa que todos los puntos están cerca de la recta?
3. En **Coefficients**, lee el signo de **Agriculture**. En `confint()`, ¿su intervalo excluye cero? Mira también el **R²**: ¿una pendiente distinta de cero garantiza que la recta ajuste bien?

In [ ]:
# Usa residuals() para obtener las diferencias observado menos ajustado.
# e_agricultura <- residuals(___)
# which.min(e_agricultura)
# which.max(e_agricultura)
# Busca esas filas en suiza para identificar los distritos.

In [ ]:
# Ejecuta después de ajustar modelo_agricultura.
# plot(modelo_agricultura, which = 1)
# X: fecundidad ajustada. Y: residuo. La línea horizontal marca cero.
# ¿Ves una curva, un abanico o algún distrito muy alejado de cero?

⏸️ **Pausa:** compara el signo de las pendientes y los $R^2$. ¿Cuál de las dos rectas representa más variación en estos datos? Recuerda que son dos modelos separados: ninguno ajusta por el predictor del otro.

### 🧪 Extensión opcional: ¿cuándo extrapolamos?

Este bloque se puede saltar en clase. Sirve para ver que una recta puede dar un número aunque no tengamos datos para respaldarlo. Compara 10 %, 50 % y 90 % con el rango de escolaridad que obtuviste con `range()`. ¿Cuál queda fuera?

In [ ]:
# Compara los tres valores propuestos con el mínimo y el máximo observados.
range(suiza$Education)
nuevos <- data.frame(Education = c(10, 50, 90))
# Una vez completado el modelo, quita el primer # para ver qué estima la recta.
# predict(modelo_educacion, newdata = nuevos)
# ¿Qué estimación es una extrapolación? Explica por qué.

## 🤝 Actividad en duplas

**Pregunta:** ¿cómo se relaciona `InfantMortality` con `Fertility`? En cada celda, el ejemplo con `#` muestra cómo se hizo el paso anterior con escolaridad. Úsalo como referencia para trabajar con mortalidad infantil. La actividad no requiere hacer predicciones.

### 📈 1. Dibuja la nube

Pon la mortalidad infantil en el eje X y la fecundidad en el eje Y. Describe si la nube sube o baja, qué forma tiene y si hay algún punto que te llame la atención.

In [ ]:
# Ejemplo anterior: ggplot(suiza, aes(x = Education, y = Fertility)) + geom_point()
# Ahora grafica mortalidad infantil en X y fecundidad en Y.

# Completa los espacios y quita el # para ejecutar.
# ggplot(suiza, aes(x = ___, y = ___)) + geom_point()
# Lo que veo en el gráfico:

### 🔢 2. Calcula Pearson y Spearman

Calcula ambos coeficientes. ¿Coinciden en la dirección? ¿Cuál destacarías en un informe y qué parte del gráfico respalda tu elección?

In [ ]:
# Ejemplo anterior: cor(suiza$Education, suiza$Fertility, method = "pearson")
# Ahora calcula Pearson y Spearman con el nuevo predictor.

# cor(suiza$InfantMortality, suiza$Fertility, method = "___")
# cor(suiza$InfantMortality, suiza$Fertility, method = "___")
# Elegiría… porque…

### 📐 3. Ajusta e interpreta la recta

Usa la fecundidad como resultado. Reporta el R² y explica cuánto cambia la fecundidad estimada cuando la mortalidad infantil sube un punto porcentual.

In [ ]:
# Ejemplo anterior: modelo_educacion <- lm(Fertility ~ Education, data = suiza)
# Ejemplo anterior: summary(modelo_educacion)
# Ejemplo anterior: confint(modelo_educacion)
# Ahora ajusta la recta con mortalidad infantil. Lee la pendiente y el R².

# modelo_infantil <- lm(___ ~ ___, data = suiza)
# summary(modelo_infantil)
# Por cada punto porcentual adicional…
# R²:

### ✍️ 4. Redacta tu conclusión

Escribe cuatro oraciones. Puedes escribirlas como comentarios de R: deja el `#` al inicio de cada línea.

In [ ]:
# Ejemplo anterior: más escolaridad se asoció con menor fecundidad estimada.
# Ahora escribe cuatro oraciones sobre mortalidad infantil y fecundidad.

# 1. Nuestra pregunta fue…
# 2. En el gráfico observamos…
# 3. Según el modelo estimamos que…
# 4. Un límite de esta interpretación es…

🎟️ **Para salir:** explica en una oración la diferencia entre correlación, pendiente y predicción. Nombra una variable que investigarías antes de plantear una explicación causal.

## 📚 Fuentes

- [Documentación de R: Swiss Fertility and Socioeconomic Indicators (1888)](https://stat.ethz.ch/R-manual/R-devel/library/datasets/html/swiss.html).